In [1]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver


In [2]:
load_dotenv()

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature=0
)

In [3]:
class JokeState(TypedDict):
    topic : str
    joke: str
    explanation : str

In [9]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [10]:
def generate_explanation(state: JokeState):
    prompt = f'generate the explanation of this joke{state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [11]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation' , generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [12]:
config1 = {"configurable": {"thread_id":"1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their quarterback!',
 'explanation': 'This joke plays on a clever pun involving the term "quarterback." In American football, a quarterback is a key player who leads the team\'s offense and is responsible for throwing passes and making strategic plays. The humor arises from the double meaning of "quarterback." \n\nIn the context of the joke, the football team is humorously portrayed as needing to go to the bank to "get their quarterback," suggesting they are looking for money (a "quarter" being a coin worth 25 cents) rather than the player. The punchline relies on the unexpected twist that combines sports terminology with banking language, creating a light-hearted and amusing scenario.'}